In [ ]:
import os
import pandas as pd
from datasets import load_dataset
from tqdm import tqdm
tqdm.pandas()

# Retrieved from: https://amazon-reviews-2023.github.io/
# NOTE: Use 2023 dataset as it is larger, more descriptive, more granular and cleaner compared to
#       previous datasets.
dataset_name = "McAuley-Lab/Amazon-Reviews-2023"

# NOTE: There are 33 categories + Unknown. Software is a subset.
category = "Software"

c:\Users\user\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


### Load User Reviews

In [ ]:
#######################################################################################################
# Data Fields for User Reviews
#######################################################################################################
#
# - rating (float)             -->   Rating of the product (from 1.0 to 5.0)
# - title (str)                -->   Title of the user review
# - text (str)                 -->   Text body of the user review
# - images (list)              -->   Images that users post after they have received the product
#                                    Each image has different sizes (small, medium, large), represented
#                                    by the small_image_url, medium_image_url, and large_image_url
#                                    respectively
# - asin (str)                 -->   ID of the product
# - parent_asin (str)          -->   Parent ID of the product. Note: Products with different colors,
#                                    styles, sizes usually belong to the same parent ID. The "asin"
#                                    in previous Amazon datasets is actually parent ID. Please use
#                                    parent ID to find product meta
# - user_id (str)              -->   ID of the reviewer
# - timestamp (int)            -->   Time of the review (unix time)
# - verified_purchase (bool)   -->   User purchase verification
# - helpful_vote (int)         -->   Helpful votes of the review
#
#######################################################################################################

# Load directly from parquet files to avoid deprecated dataset script
review_dataset = load_dataset(
    "parquet",
    data_files=f"hf://datasets/{dataset_name}/raw_review_{category}/full/0000.parquet",
    split="train"
)

all_reviews = [review for review in tqdm(review_dataset)]
review_df = pd.DataFrame(all_reviews)

review_df

### Load Item Metadata

In [ ]:
#######################################################################################################
# Data Fields for Item Metadata
#######################################################################################################
#
# - main_category (str)	       -->   Main category (i.e., domain) of the product
# - title (str)	               -->   Name of the product
# - average_rating (float)     -->   Rating of the product shown on the product page
# - rating_number (int)        -->   Number of ratings in the product
# - features (list)            -->   Bullet-point format features of the product
# - description (list)         -->   Description of the product
# - price (float)              -->   Price in US dollars (at time of crawling)
# - images (list)              -->   Images of the product. Each image has different sizes (thumb, large,
#                                    hi_res). The "variant" field shows the position of image
# - videos (list)              -->   Videos of the product including title and url
# - store (str)	               -->   Store name of the product
# - categories (list)	       -->   Hierarchical categories of the product
# - details (dict)             -->   Product details, including materials, brand, sizes, etc
# - parent_asin (str)	       -->   Parent ID of the product
# - bought_together (list)     -->   Recommended bundles from the websites
#
#######################################################################################################

# Load directly from parquet files to avoid deprecated dataset script
meta_dataset = load_dataset(
    "parquet",
    data_files=f"hf://datasets/{dataset_name}/raw_meta_{category}/full/0000.parquet",
    split="train"
)

meta_df = pd.DataFrame(meta_dataset)

meta_df

### Export Dataset

In [5]:
output_dir = "./output"

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Export DataFrame
review_df.to_parquet(f"{output_dir}/review_data.parquet", index=False)
meta_df.to_parquet(f"{output_dir}/meta_data.parquet", index=False)